In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Output
from IPython.display import display
from pathlib import Path
import io, warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

DATA_DIR = Path('../data')

In [2]:
# ── Load data ─────────────────────────────────────────────────────────────────
COLS = [
    'ID_CONTRACT', 'VEHICLE_ID', 'ID_QUOTATION', 'COB_DATE',
    'COUNTRY', 'BRAND_UPDATE', 'POWER_CATEGORY', 'VA_CO2_EMSS_REAL',
    'NOVA_ASSET_STATUS', 'BIKE_OR_CAR',
    'OBLIGOR_IDENTIFIER', 'GROUP_RATING', 'COUNTERPARTY_RATING',
    'CLS_GROUP_RATING', 'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION',
    'SHARED_CLIENT_FLAG', 'VEHICLE_PRICE_EUR',
    'EXPOSURE_AMOUNT_LTR', 'EXPOSURE_AMOUNT_MTR',
]

files = sorted(DATA_DIR.glob('NOVA - *.parquet'))
print(f'Loading {len(files)} files...')
nova = pd.concat([pd.read_parquet(f, columns=COLS) for f in files], ignore_index=True)

nova['EXPOSURE_AMOUNT_TOT'] = nova['EXPOSURE_AMOUNT_LTR'] + nova['EXPOSURE_AMOUNT_MTR']

def co2_bucket(v):
    try:
        n = int(float(v)); lo = (n // 10) * 10
        return f'[{lo}-{lo+9}]'
    except: return 'UNK'
nova['CO2_BUCKET'] = nova['VA_CO2_EMSS_REAL'].apply(co2_bucket)

# Keep latest snapshot per contract
nova = (nova.sort_values('COB_DATE')
            .drop_duplicates(subset=['ID_CONTRACT','VEHICLE_ID','ID_QUOTATION'], keep='last'))

ALL_BRANDS    = sorted(nova['BRAND_UPDATE'].dropna().unique())
ALL_COUNTRIES = sorted(nova['COUNTRY'].dropna().unique())

print(f'Ready: {len(nova):,} rows | {len(ALL_BRANDS)} brands | {len(ALL_COUNTRIES)} countries')

Loading 1056 files...
Ready: 1,124,965 rows | 26 brands | 8 countries


## Global filters — applied to ALL sections below

In [3]:
# ── Global filters (shared across all cells) ──────────────────────────────────
_style = {'description_width': '60px'}
_layout = widgets.Layout(width='220px')

gf_brand   = widgets.Dropdown(options=['All'] + ALL_BRANDS,    value='All',
                               description='Brand:',   style=_style, layout=_layout)
gf_country = widgets.Dropdown(options=['All'] + ALL_COUNTRIES, value='All',
                               description='Country:', style=_style, layout=_layout)

def apply_global_filters(df):
    out = df
    if gf_brand.value   != 'All': out = out[out['BRAND_UPDATE'] == gf_brand.value]
    if gf_country.value != 'All': out = out[out['COUNTRY']      == gf_country.value]
    return out

display(widgets.HTML('<b>Global filters</b> (apply to heatmap AND simulation):'))
display(HBox([gf_brand, gf_country]))

HTML(value='<b>Global filters</b> (apply to heatmap AND simulation):')

## Section 1 — Interactive Heatmap

In [4]:
# ── Utilities ─────────────────────────────────────────────────────────────────
RATING_COLS = {'CLS_GROUP_RATING', 'COUNTERPARTY_RATING', 'GROUP_RATING'}

def fmt(val):
    if pd.isna(val) or val == 0: return ''
    if val >= 1_000_000: return f'{val/1_000_000:.1f}M'
    if val >= 1_000:     return f'{val/1_000:.1f}k'
    return str(int(val))

def get_pivot(df, y_col, x_cols, metric):
    x_cols = list(x_cols)
    has_rating = bool(set(x_cols) & RATING_COLS)
    keys = (['OBLIGOR_IDENTIFIER'] if has_rating else []) + \
           ['ID_CONTRACT','VEHICLE_ID','ID_QUOTATION']
    keys = [k for k in keys if k in df.columns]

    df2 = df.dropna(subset=[y_col] + x_cols) \
             .drop_duplicates(subset=keys + [y_col] + x_cols)
    if df2.empty: return pd.DataFrame()

    g = [y_col] + x_cols
    if metric == 'concentration_financiere':
        agg = df2.groupby(g)['EXPOSURE_AMOUNT_TOT'].sum().reset_index(name='v')
    elif metric == 'intensite_risk_asset':
        agg = df2.groupby(g)['VEHICLE_PRICE_EUR'].sum().reset_index(name='v')
    else:
        agg = df2.groupby(g).size().reset_index(name='v')

    # Combined X label = actual values (field name as axis label set after)
    agg['_x'] = agg[x_cols].astype(str).agg(' | '.join, axis=1)
    pivot = agg.pivot_table(index=y_col, columns='_x', values='v',
                             aggfunc='sum', fill_value=0)
    pivot = pivot[pivot.sum().sort_values(ascending=False).index]  # sort cols by total
    pivot.columns.name = ' | '.join(x_cols)  # <-- show real field name on axis
    return pivot


def draw_heatmap(pivot, title, metric, page, cmap, ax):
    rows_pp = 25
    sub = pivot.iloc[page * rows_pp : (page + 1) * rows_pp]
    if sub.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        return
    annot = sub.map(fmt)
    sns.heatmap(sub, annot=annot, fmt='', cmap=cmap,
                annot_kws={'size': 8}, linewidths=0.25, linecolor='#dddddd',
                ax=ax)
    ax.set_title(title, fontsize=10, pad=6)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)


# ── Section 1 UI ──────────────────────────────────────────────────────────────
Y_OPTS = ['BRAND_UPDATE','POWER_CATEGORY','CO2_BUCKET',
          'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION','SHARED_CLIENT_FLAG']
X_OPTS = ['GROUP_RATING','COUNTERPARTY_RATING','CLS_GROUP_RATING',
          'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION','SHARED_CLIENT_FLAG','POWER_CATEGORY']
M_OPTS = [('Volume (nb contrats)','volume'),
          ('Concentration Financière','concentration_financiere'),
          ('Intensité Risk Asset','intensite_risk_asset')]

_s2 = {'description_width': '65px'}
h_y  = widgets.Dropdown(options=Y_OPTS, value='BRAND_UPDATE', description='Axe Y:', style=_s2)
h_x  = widgets.SelectMultiple(options=X_OPTS, value=['GROUP_RATING'],
                               description='Axe X:', rows=5, style=_s2)
h_m  = widgets.Dropdown(options=M_OPTS, description='Métrique:', style=_s2)
h_pg = widgets.IntSlider(min=0, max=0, step=1, value=0, description='Page:',
                          continuous_update=False, layout=widgets.Layout(width='300px', display='none'))
h_out = Output()

def h_refresh(*_):
    df_f  = apply_global_filters(nova)
    pivot = get_pivot(df_f, h_y.value, list(h_x.value), h_m.value)

    n_pages = max(0, (len(pivot) - 1) // 25)
    h_pg.max = n_pages
    h_pg.layout.display = 'flex' if n_pages > 0 else 'none'

    with h_out:
        h_out.clear_output(wait=True)
        if pivot.empty:
            print('No data for this selection.')
            return
        n_cols = min(len(pivot.columns), 20)
        height = max(6, 25 * 0.4)
        fig, ax = plt.subplots(figsize=(max(14, n_cols * 1.1), height))
        draw_heatmap(pivot,
                     f'{h_y.value}  ×  {", ".join(h_x.value)}  —  {dict(M_OPTS).get(h_m.value, h_m.value)}',
                     h_m.value, h_pg.value, 'YlGnBu', ax)
        plt.tight_layout()
        display(fig)
        plt.close(fig)

for w in [h_y, h_x, h_m, h_pg, gf_brand, gf_country]:
    w.observe(h_refresh, names='value')

display(VBox([
    HBox([h_y, h_x, h_m]),
    h_pg,
    h_out
]))
h_refresh()

## Section 2 — Simulation

In [ ]:
# ── Simulation helpers ────────────────────────────────────────────────────────
SIM_RNG = np.random.default_rng(42)

def empirical_rating_dist(df, y_col, y_val):
    sub = df[df[y_col].astype(str) == str(y_val)]['GROUP_RATING'].dropna()
    if sub.empty: return None, None
    c = sub.value_counts(normalize=True)
    return c.index.tolist(), c.values.tolist()


def sample_rows(df, y_col, y_val, n):
    tmpl = df[df[y_col].astype(str) == str(y_val)]
    if tmpl.empty or n == 0: return pd.DataFrame()
    sampled = tmpl.sample(n=n, replace=True, random_state=int(SIM_RNG.integers(1_000_000))).copy()
    for col in ['ID_CONTRACT','VEHICLE_ID','ID_QUOTATION','OBLIGOR_IDENTIFIER']:
        if col in sampled.columns:
            sampled[col] = [f'SIM_{i}' for i in range(n)]
    vals, probs = empirical_rating_dist(df, y_col, y_val)
    if vals:
        sampled['GROUP_RATING'] = SIM_RNG.choice(vals, size=n, p=probs)
    if 'VEHICLE_PRICE_EUR' in sampled.columns:
        std = max(float(tmpl['VEHICLE_PRICE_EUR'].std()), 500)
        sampled['VEHICLE_PRICE_EUR'] = np.clip(
            sampled['VEHICLE_PRICE_EUR'] + SIM_RNG.normal(0, std * 0.1, n), 1000, None
        ).round(2)
        sampled['EXPOSURE_AMOUNT_LTR'] = (sampled['VEHICLE_PRICE_EUR'] * SIM_RNG.uniform(0.4, 0.65, n)).round(2)
        sampled['EXPOSURE_AMOUNT_MTR'] = (sampled['VEHICLE_PRICE_EUR'] * SIM_RNG.uniform(0.05, 0.15, n)).round(2)
        sampled['EXPOSURE_AMOUNT_TOT'] = sampled['EXPOSURE_AMOUNT_LTR'] + sampled['EXPOSURE_AMOUNT_MTR']
    return sampled


def build_sim_df(df_orig, y_col, add_vals, rem_vals):
    """add_vals / rem_vals are {str: int} dicts."""
    df2 = df_orig.copy()
    for val, n in rem_vals.items():
        if n <= 0: continue
        idx = df2[df2[y_col].astype(str) == str(val)].index.to_numpy()
        n = min(n, len(idx))
        if n > 0:
            drop_idx = SIM_RNG.choice(idx, size=n, replace=False)
            df2 = df2.drop(index=drop_idx)
    new_rows = [sample_rows(df_orig, y_col, val, n)
                for val, n in add_vals.items() if n > 0]
    non_empty = [r for r in new_rows if not r.empty]
    if non_empty:
        df2 = pd.concat([df2] + non_empty, ignore_index=True)
    return df2


def plot_sim_panels(df_orig, df_sim, y_col, x_cols, metric, metric_label, add_vals, rem_vals):
    """Original | Simulated on top row, Delta full-width below."""
    p_orig = get_pivot(df_orig, y_col, x_cols, metric)
    p_sim  = get_pivot(df_sim,  y_col, x_cols, metric)
    if p_orig.empty and p_sim.empty:
        print('No data for this selection.'); return

    all_idx  = sorted(set(p_orig.index)  | set(p_sim.index))
    all_cols = list(p_orig.columns.union(p_sim.columns))
    orig  = p_orig.reindex(index=all_idx, columns=all_cols, fill_value=0)
    sim   = p_sim.reindex( index=all_idx, columns=all_cols, fill_value=0)
    delta = sim - orig

    top_n = orig.sum().sort_values(ascending=False).head(18).index
    orig, sim, delta = orig[top_n], sim[top_n], delta[top_n]

    n_r     = len(all_idx)
    panel_h = max(5, min(n_r, 25) * 0.42)
    col_w   = max(14, len(top_n) * 0.95)

    fig = plt.figure(figsize=(col_w * 2 + 1, panel_h * 2 + 1.5), facecolor='white')
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.6, wspace=0.06,
                             height_ratios=[1, 1])
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, :])

    kw = dict(fmt='', annot_kws={'size': 8}, linewidths=0.25, linecolor='#e0e0e0')

    sns.heatmap(orig,  annot=orig.map(fmt),  cmap='Blues',  ax=ax1, **kw)
    ax1.set_title('Portefeuille Actuel', fontsize=11, fontweight='bold')
    ax1.tick_params(axis='x', rotation=45, labelsize=7)
    ax1.tick_params(axis='y', rotation=0,  labelsize=8)

    sns.heatmap(sim,   annot=sim.map(fmt),   cmap='BuGn',   ax=ax2, **kw)
    ax2.set_title('Portefeuille Simulé', fontsize=11, fontweight='bold', color='#1a7d44')
    ax2.set_ylabel('')
    ax2.set_yticklabels([])
    ax2.tick_params(axis='x', rotation=45, labelsize=7)

    vmax = max(float(np.abs(delta.values).max()), 1)
    sns.heatmap(delta, annot=delta.map(fmt), cmap='RdYlGn',
                center=0, vmin=-vmax, vmax=vmax, ax=ax3, **kw)
    ax3.set_title('Delta  (Simulé − Actuel)     ▲ vert = hausse     ▼ rouge = baisse',
                  fontsize=11, fontweight='bold', color='#555')
    ax3.tick_params(axis='x', rotation=45, labelsize=7)
    ax3.tick_params(axis='y', rotation=0,  labelsize=8)

    n_add = sum(v for v in add_vals.values() if v > 0)
    n_rem = sum(v for v in rem_vals.values() if v > 0)
    fig.suptitle(
        f'{y_col}  ×  {", ".join(x_cols)}   |   {metric_label}\n'
        f'Portfolio : {len(df_orig):,} → {len(df_sim):,} contrats'
        f'   (+{n_add} ajoutés   −{n_rem} retirés)',
        fontsize=11, y=1.01
    )
    plt.tight_layout()
    display(fig)
    plt.close(fig)


# ── Simulation UI ─────────────────────────────────────────────────────────────
PAGE_SIZE = 15
add_d, rem_d = {}, {}   # {str: BoundedIntText}

s_y  = widgets.Dropdown(
    options=['BRAND_UPDATE','POWER_CATEGORY','CO2_BUCKET','ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION'],
    value='BRAND_UPDATE', description='Simuler sur:',
    style={'description_width': '95px'}, layout=widgets.Layout(width='290px'))
s_x  = widgets.SelectMultiple(
    options=['GROUP_RATING','COUNTERPARTY_RATING','CLS_GROUP_RATING',
             'ARVAL_INDUSTRY_CODE_CLS_DESCRIPTION','SHARED_CLIENT_FLAG'],
    value=['GROUP_RATING'], rows=5, description='Axe X:',
    style={'description_width': '60px'})
s_m  = widgets.Dropdown(
    options=[('Volume','volume'),
             ('Concentration Financière','concentration_financiere'),
             ('Intensité Risk Asset','intensite_risk_asset')],
    description='Métrique:', style={'description_width': '70px'})
s_pg = widgets.BoundedIntText(
    value=1, min=1, max=1, description='Page:',
    layout=widgets.Layout(width='120px'), style={'description_width': '45px'})

run_btn   = widgets.Button(description='Run Simulation', button_style='success',
                            icon='play',    layout=widgets.Layout(width='180px'))
reset_btn = widgets.Button(description='Reset',          button_style='warning',
                            icon='refresh', layout=widgets.Layout(width='100px'))

ctrl_box = widgets.VBox([])
s_out    = Output()


def build_ctrl_rows(y_col, page):
    global add_d, rem_d
    df_f   = apply_global_filters(nova)
    y_vals = sorted(df_f[y_col].dropna().astype(str).unique())
    total  = max(1, -(-len(y_vals) // PAGE_SIZE))
    s_pg.max = total

    start     = (page - 1) * PAGE_SIZE
    page_vals = y_vals[start: start + PAGE_SIZE]

    add_d, rem_d = {}, {}
    rows = []
    for val in page_vals:
        n_cur = int((df_f[y_col].astype(str) == val).sum())
        add_w = widgets.BoundedIntText(value=0, min=0, max=50_000, step=10,
                                        layout=widgets.Layout(width='100px'))
        rem_w = widgets.BoundedIntText(value=0, min=0, max=n_cur,  step=10,
                                        layout=widgets.Layout(width='100px'))
        add_d[val] = add_w
        rem_d[val] = rem_w
        lbl = widgets.HTML(
            f'<span style="display:inline-block;min-width:190px;font-weight:bold">{val}</span>'
            f'<span style="color:#888;font-size:12px"> ({n_cur:,} contrats)</span>'
        )
        rows.append(HBox([lbl, widgets.Label('＋ Add:'), add_w,
                          widgets.Label('－ Remove:'), rem_w]))

    header = widgets.HTML(
        f'<hr><b>Ajuster les volumes — {y_col}</b> '
        f'(page {page}/{total}, {len(y_vals)} valeurs total)<br>'
        '<span style="font-size:11px;color:#666">'
        'Les attributs des véhicules ajoutés sont échantillonnés depuis la distribution empirique de la catégorie.'
        '</span>'
    )
    ctrl_box.children = [header] + rows + [HBox([run_btn, reset_btn])]


def on_y_or_page(*_):
    build_ctrl_rows(s_y.value, s_pg.value)


def on_run(_):
    df_f      = apply_global_filters(nova)
    y_col     = s_y.value
    add_vals  = {v: w.value for v, w in add_d.items()}
    rem_vals  = {v: w.value for v, w in rem_d.items()}
    df_sim    = build_sim_df(df_f, y_col, add_vals, rem_vals)
    metric    = s_m.value
    m_label   = dict(s_m.options).get(metric, metric)
    with s_out:
        s_out.clear_output(wait=True)
        plot_sim_panels(df_f, df_sim, y_col, list(s_x.value), metric, m_label, add_vals, rem_vals)


def on_reset(_):
    for w in list(add_d.values()) + list(rem_d.values()):
        w.value = 0
    with s_out:
        s_out.clear_output()


s_y.observe(on_y_or_page,  names='value')
s_pg.observe(on_y_or_page, names='value')
gf_brand.observe(on_y_or_page,   names='value')
gf_country.observe(on_y_or_page, names='value')
run_btn.on_click(on_run)
reset_btn.on_click(on_reset)

display(VBox([HBox([s_y, s_x, s_m, s_pg]), ctrl_box, s_out]))
build_ctrl_rows(s_y.value, s_pg.value)
